In [4]:
#this script reads all CSV files in the curated-data directory and prints their shape and the number of null values per column.
import pandas as pd
from pathlib import Path

data_dir = Path("../curated-data")
csv_files = sorted(data_dir.glob("*.csv"))

for file in csv_files:
    df = pd.read_csv(file)
    print(f"--- {file.name} ---")
    print("Shape:", df.shape)
    print("Nulls per column:\n", df.isnull().sum())
    print()

--- CostCategory.csv ---
Shape: (5, 3)
Nulls per column:
 id       0
code     0
label    0
dtype: int64

--- Country.csv ---
Shape: (3, 3)
Nulls per column:
 id          0
name        0
iso_code    0
dtype: int64

--- Currency.csv ---
Shape: (3, 3)
Nulls per column:
 id          0
iso_code    0
symbol      0
dtype: int64

--- Destination.csv ---
Shape: (3, 7)
Nulls per column:
 id              0
country_id      0
name            0
description     0
latitude        0
longitude       0
is_supported    0
dtype: int64

--- Extra_AI_Context.csv ---
Shape: (57, 7)
Nulls per column:
 place_id                 0
place_name               0
budget_tier              0
suitable_travel_style    9
interest_tag             9
source_url               0
notes                    0
dtype: int64

--- InterestCategory.csv ---
Shape: (8, 3)
Nulls per column:
 id       0
code     0
label    0
dtype: int64

--- Place.csv ---
Shape: (57, 10)
Nulls per column:
 id                   0
destination_id       0
place

In [ ]:
#this script reads the DATASET_MANIFEST.json file and compares the number of rows in each CSV file with the expected number of rows in the manifest.

import json

with open("../seed/DATASET_MANIFEST.json") as f:
    manifest = json.load(f)

print("Comparing row counts vs manifest:\n")
all_match = True
for file in csv_files:
    df = pd.read_csv(file)
    expected = manifest["files"].get(file.name, {}).get("rows")
    actual = len(df)
    status = "✅" if actual == expected else "❌"
    if actual != expected:
        all_match = False
    print(f"{status} {file.name}: actual={actual}, expected={expected}")

print("\nALL MATCH" if all_match else "\nMISMATCH FOUND")

Comparing row counts vs manifest:

✅ CostCategory.csv: actual=5, expected=5
✅ Country.csv: actual=3, expected=3
✅ Currency.csv: actual=3, expected=3
✅ Destination.csv: actual=3, expected=3
✅ Extra_AI_Context.csv: actual=57, expected=57
✅ InterestCategory.csv: actual=8, expected=8
✅ Place.csv: actual=57, expected=57
✅ PlaceCategory.csv: actual=5, expected=5

ALL MATCH


In [6]:
import csv

path = "../curated-data/Extra_AI_Context.csv"
with open(path, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    rows = list(reader)

transport_ids = {"17","18","19","36","37","38","55","56","57"}
note = "interest_tag intentionally blank — TRANSPORT/logistics place, not interest-relevant by design (excluded from PlaceInterest scope)"

for row in rows:
    if row["place_id"] in transport_ids:
        row["notes"] = (row["notes"] + " | " if row["notes"] else "") + note

with open(path, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("Updated", len(transport_ids), "rows")

Updated 9 rows
